Cài đặt streamlit

In [12]:
pip install -q streamlit pypdf chromadb ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/1

cài đặt ollama

In [42]:
#cài các thư viện cần
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw

#cài ollama
!curl -fsSL https://ollama.com/install.sh | sh

#tắt Flash Attention để tránh lỗi tương thích phần cứng trên colab
import os, subprocess, time
os.environ["OLLAMA_FLASH_ATTENTION"] = "false"

#khởi chạy ollama server ở chế độ nền
subprocess.Popen(["ollama", "serve"])
time.sleep(30)

#tải 2 model cần dùng
!ollama pull vicuna:7b-v1.5-q5_1
!ollama pull bge-m3

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+

import và cấu hình

In [33]:
%%writefile app.py
import streamlit as st
import tempfile, os, time
import pypdf
import chromadb
import ollama

#tên model gán thành biến để sau này dễ đổi
LLM_MODEL ="vicuna:7b-v1.5-q5_1"
EMBED_MODEL ="bge-m3"

PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh: {context}

Câu hỏi: {question}
Trả lời:"""

Overwriting app.py


khởi tạo session state

In [34]:
%%writefile -a app.py
#ta muốn lưu dữ liệu lại sau mỗi lần tương tác với web: giữ lại vector database, tên file đang xử lý và lịch sử chat
for k, v in {"collection": None, "pdf_name": "", "chat_history": []}.items():
 st.session_state.setdefault(k,v)

Appending to app.py


các hàm sử lý ⁉

- hàm embed và chunk_text giữ nguyên

- hàm rag được điều chình: Nhận thêm tham số collection và tích hợp truy vấn chromaDB thay vì gọi hàm retrieve() riêng

- hàm process_pdf thêm logic xử lý upload file từ treamlit: up loadfile lên thì file chỉ tồn tại trông bộ nhớ -> lưu ra file thật để đọc và đọc xong thì giải phóng bộ nhớ




In [35]:
%%writefile -a app.py

def embed(texts):
  #chuyển text thành vector embedding
  return ollama.embed(model=EMBED_MODEL, input = texts)["embeddings"]

def chunk_text(text, size = 1000, overlap=200):
  #cắt text thành các chunk nhỏ
  paras = [p.strip() for p in text.split("\n") if p.strip()]
  chunks, cur = [], ""
  for p in paras:
    if len(cur) + len(p) + 1 <= size:
      cur += p + "\n"
    else:
      if cur:
        chunks.append(cur.strip())
      cur = (cur[-overlap:] + p + "\n") if overlap else (p + "\n")
  if cur.strip():
    chunks.append(cur.strip())
  return chunks

def process_pdf(uploaded_file):
  #đọc file, cắt nhỏ, embedding và lưu cào chromaDB
  #lưu upload file thành file tạm
  with tempfile.NamedTemporaryFile(delete = False, suffix = ".pdf") as tmp:
    tmp.write(uploaded_file.getvalue())
    path = tmp.name

  #đọc nội dung file
  text ="\n".join(p.extract_text() or "" for p in pypdf.PdfReader(path).pages)
  os.unlink(path) #xóa file tạm

  #cắt nhỏ và lưu vàu chromaDB
  chunks = chunk_text(text)
  client = chromadb.Client()
  col = client.get_or_create_collection(f"rag_{int(time.time())}")
  col.add(
      ids=[str(i) for i in range(len(chunks))],
      documents= chunks,
      embeddings= embed(chunks)
  )
  return col, len(chunks)

def rag(question, collection, k=4):
  #hàm RAG: tìm context và hỏi LLM
  res = collection.query(query_embeddings = embed([question]), n_results=k)
  context = "\n\n".join(res["documents"][0])
  resp = ollama.chat(
      model = LLM_MODEL,
      messages = [{"role": "user", "content": PROMPT.format(context=context, question=question)}]
  )
  return resp["message"]["content"]

Appending to app.py


Giao diện streamlit

In [36]:
%%writefile -a app.py

# cấu hình trang
st.set_page_config(page_title = "PDF RAG Chatbot", layout = "wide", initial_sidebar_state = "expanded")
st.title("PDF RAG Assistant: Native")

#sidebar: uploaf PDF và nút điều khiển
with st.sidebar:
  st.subheader("Upload file")
  f = st.file_uploader("Chọn file", type = "pdf")
  if f and st.button("Xử lý file", use_container_width=True):
    with st.spinner("Đang xử lý..."):
      st.session_state.collection, n = process_pdf(f)
      st.session_state.pdf_name = f.name
      st.session_state.chat_history = []
    st.success(f"{n} chunks")
  st.info(f"{st.session_state.pdf_name}" if st.session_state.pdf_name
          else "Chưa có file")
  if st.button("Xóa lịch xử chat", use_container_width=True):
    st.session_state.chat_history = []

#hiển thị lịch sử chat
for m in st.session_state.chat_history:
  with st.chat_message(m["role"]):
    st.write(m["content"])

#ô nhập input
if st.session_state.collection is None:
  st.info("Upload và xử lý file trước")
  st.chat_input("Nhập câu hỏi...", disabled =True)
else:
   q = st.chat_input("Nhập câu hỏi...")
   if q:
    st.session_state.chat_history.append({"role":"user", "content":q})
    with st.chat_message("user"):
      st.write(q)
    with st.chat_message("assistant"):
      with st.spinner("Đang suy nghĩ..."):
        ans = rag(q, st.session_state.collection)
        st.write(ans)
    st.session_state.chat_history.append({"role": "assistant", "content": ans})

Appending to app.py


!cat app.py

In [37]:
!cat app.py

import streamlit as st
import tempfile, os, time
import pypdf
import chromadb
import ollama

#tên model gán thành biến để sau này dễ đổi
LLM_MODEL ="vicuna:7b-v1.5-q5_1"
EMBED_MODEL ="bge-m3"

PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
11 Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
12 Trả lời ngắn gọn, chính xác, bằng tiếng Việt.
13
14 Ngữ cảnh: {context}
15
16 Câu hỏi: {question}
17 Trả lời:"""
#ta muốn lưu dữ liệu lại sau mỗi lần tương tác với web: giữ lại vector database, tên file đang xử lý và lịch sử chat
for k, v in {"collection": None, "pdf_name": "", "chat_history": []}.items():
 st.session_state.setdefault(k,v)

def embed(texts):
  #chuyển text thành vector embedding
  return ollama.embed(model=EMBED_MODEL, input = texts)["embeddings"]

def chunk_text(text, size = 1000, overlap=200):
  #cắt text thành các chunk nhỏ
  paras = [p.strip() for p in text.split("\n") if p.strip()]
  chunks, cur = [], ""
  for p in

chạy ứng dụng

In [ ]:
#tải clouflared (chạy 1 lần)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O clouflared

!chmod +x clouflared

#chạy streamlit nền + mở tunnel
!streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false &>/content/st.log &

import time; time.sleep(8)

!./clouflared tunnel --url http://localhost:8501

2026-06-27T03:34:05Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-27T03:34:05Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-27T03:34:08Z INF +--------------------------------------------------------------------------------------------+
2026-06-27T03:34:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-27T03:34:08Z INF |  https://collection-broke-naturals-magazines.trycloudf